# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%cd /content/flyrank-ml-internship-laiba_sabir

/content/flyrank-ml-internship-laiba_sabir


In [ ]:
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

print("Current directory:", Path.cwd())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH.resolve())

Current directory: /content/flyrank-ml-internship-laiba_sabir
Dataset exists: True
Dataset path: /content/flyrank-ml-internship-laiba_sabir/data/raw/content_refresh_anonymized.csv


In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

FIG_DIR = "work/figures"
OUT_DIR = "work/outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"{len(df):,} rows x {df.shape[1]} columns | {df['client_id'].nunique()} clients")
print(f"Declining-label rate: {df['is_declining_label'].mean():.3f}")


30,000 rows x 45 columns | 32 clients
Declining-label rate: 0.542


## 1. Question

*The research question and the decision it supports.*

**Research question:** Which of ~30,000 pages should a content reviewer look at first this
week, given capacity for only ~50 pages?

**Decision improved:** the weekly triage step — right now a reviewer either works through pages
in an arbitrary order or leans on a hand-written product rule (`health_score`, `needs_ctr_fix`).
This capstone replaces "arbitrary or hand-tuned" with "ranked by evidence."

**Who acts on the output:** a content reviewer with fixed weekly capacity. They open the ranked
queue, work top to bottom, and stop when capacity runs out.

**Cost of a wrong call:** two directions matter. A **false positive** near the top wastes a
reviewer's limited hour on a page that wasn't actually a priority. A **false negative** — a
genuinely declining, high-traffic page ranked low — quietly keeps losing visibility for another
week before anyone notices. Because capacity is fixed at ~50/week, the right metric is not
overall accuracy but **Precision@50**: of the top 50 pages the system flags, how many are
actually declining?

**Why ML earns its place here:** a single hand-written rule (stale AND visible) is easy to state
but leaves real signal on the table — position, engagement, content depth, and freshness
interact in ways that are hard to hand-tune. Section 4 shows the size of that gap directly.

**Claim discipline:** every result below is **observed / measured / decision-support**, tied to
a specific split, sample size, and this one run. Nothing here claims to predict Google's
algorithm or that a refresh *causes* recovery — proving causation would need an experiment this
data cannot give us (see `docs/ml-intern-dataset-and-lane-guide.md`, section 6).


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



**Primary source for this notebook:** the bundled anonymized starter slice,
`data/raw/content_refresh_anonymized.csv` — 30,000 pages × 44 columns, 32 pseudonymized clients,
trailing-90-day metrics as of the export date. This is the same slice used throughout the
weekly lane notebooks (`w01`–`w07`), so results here are directly comparable to that earlier
work.

**Full-scale counterpart:** the lane and validation work in `w06_validation_audit.ipynb` and
`w07_action_playbook.ipynb` also draws on the full pseudonymized warehouse release
(`FlyRank/internship-warehouse`, ~79M rows, build `flyrank_pseudonymized_warehouse_release_v20260703`,
`dim_clients` + `dim_content` + `fact_content_daily_performance`, Jan 2025–Jun 2026). That
release requires a gated Hugging Face token and is not reachable from this execution
environment, so the numbers computed live in *this* notebook run on the 30k-row starter slice;
where the two diverge, the weekly notebooks are the source of record for the warehouse-scale
number and this notebook says so explicitly.

**What was excluded, and why:**
- `content_id` / `client_id` — pseudonymous identifiers, used only for grouping and the
  client-holdout split, never as model features.
- `trend_direction` / `trend_pct` — the label is *derived from* these, so using them as features
  would be leakage (demonstrated live in Section 3).
- FlyRank's own product decision flags (`health_score`, `needs_ctr_fix`, `is_quick_win`,
  `priority_score`, `action_type`) were never shipped in this dataset in the first place — the
  starter slice ships observable signals only, by design.
- No client names, domains, URLs, page titles, or raw keywords appear anywhere in the data or
  in this notebook's output.

**Missingness is systematic, not random** — keyword-context columns are blank for `feedly
article` rows entirely; `word_count` is blank for a meaningful slice. The prep step below fills
numerics with 0 and categoricals with `"unknown"` rather than dropping rows, and the columns
where that matters carry an explicit `_tier` bucket alongside the raw value.

In [ ]:
missing_summary = df[["search_volume", "word_count", "provider_used", "main_intent"]].isna().mean().round(3)
print("Missing-value rate on a few gotcha columns:")
print(missing_summary.to_string())

print("\nRate-column sanity check (these are already x100 percentages, per the data dictionary):")
print(df[["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]].describe().loc[["mean", "50%", "max"]].round(2))


Missing-value rate on a few gotcha columns:
search_volume    0.082
word_count       0.257
provider_used    0.715
main_intent      0.079

Rate-column sanity check (these are already x100 percentages, per the data dictionary):
         ctr  engagement_rate  scroll_rate  ai_traffic_pct
mean    0.51             2.53        18.21            0.77
50%     0.07             0.00         5.00            0.00
max   100.00           100.00       300.00          300.00


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Unit of analysis:** one row = one content page.

**Label (proxy, not a future outcome):** `is_declining_label = (trend_direction == "down")`,
where `trend_direction` compares the last-30-day window to the previous-30-day window on GSC
impressions. This is a *current-window* proxy, not a future-outcome label — a known limitation,
carried into Section 5.

**Feature set** (observable-only, pre-decision signals — no product flags, no label-derived
columns):


In [ ]:
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

for col in ["search_volume", "competition", "cpc", "word_count", "char_count",
            "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
            "days_with_impressions", "days_with_sessions", "content_age_days",
            "days_since_last_update", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "ai_traffic_pct"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

for col in MODEL_CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("unknown").astype(str)

print(f"{len(MODEL_NUMERIC_FEATURES)} numeric features, {len(MODEL_CATEGORICAL_FEATURES)} categorical features.")
print("Excluded on purpose: content_id, client_id (grouping only), trend_direction, trend_pct (label-derived).")


18 numeric features, 8 categorical features.
Excluded on purpose: content_id, client_id (grouping only), trend_direction, trend_pct (label-derived).


**Baseline (the transparent rule this model must beat):**

```text
baseline_refresh_score = 0.40 * visibility_score
                        + 0.30 * freshness_risk_score
                        + 0.25 * position_opportunity_score
                        + 0.05 * depth_gap_score
```

Each sub-score is a percentile rank of an observable signal (impressions, staleness, position,
depth) — readable by a human, no fitted weights.


In [ ]:
def percentile_rank(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def normalize(s):
    v = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo) if hi > lo else pd.Series(np.zeros(len(v)), index=v.index)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"] * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"] + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"] + 0.05 * df["depth_gap_score"]
).clip(0, 1)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

base_rate = df["is_declining_label"].mean()
baseline_p50 = precision_at_k(df["is_declining_label"], df["baseline_refresh_score"], 50)
print(f"Base rate (always-flag-majority): {base_rate:.3f}")
print(f"Baseline rule Precision@50 (full data, in-sample — see Section 4 for the honest holdout number): {baseline_p50:.3f}")


Base rate (always-flag-majority): 0.542
Baseline rule Precision@50 (full data, in-sample — see Section 4 for the honest holdout number): 0.340


**Leakage hunt — attacking our own features before trusting anything.** `trend_pct` is the
exact percentage the label is thresholded from, so it must never be a feature. Here is the
confession: fit the same model with and without it.


In [ ]:
def build_matrix(frame, extra_numeric=None):
    numeric_cols = MODEL_NUMERIC_FEATURES + (extra_numeric or [])
    num = frame[numeric_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = pd.get_dummies(frame[MODEL_CATEGORICAL_FEATURES].astype(str), dummy_na=False, dtype=float)
    return pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)

y = df["is_declining_label"].astype(int)

X_clean = build_matrix(df)
X_leaky = build_matrix(df, extra_numeric=["trend_pct"])

clean_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=RANDOM_STATE).fit(X_clean, y)
leaky_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=RANDOM_STATE).fit(X_leaky, y)

clean_p50 = precision_at_k(y, clean_tree.predict_proba(X_clean)[:, 1], 50)
leaky_p50 = precision_at_k(y, leaky_tree.predict_proba(X_leaky)[:, 1], 50)

print(f"Clean features   Precision@50 (in-sample): {clean_p50:.3f}")
print(f"+ trend_pct      Precision@50 (in-sample): {leaky_p50:.3f}   <- confession: leakage, not skill")
leak_importance = float(pd.Series(leaky_tree.feature_importances_, index=X_leaky.columns).get("trend_pct", 0.0))
print(f"trend_pct feature-importance share in the leaky model: {leak_importance:.3f}")
print("\nVerdict: trend_pct is excluded from every model below. This block exists only to prove the exclusion is necessary.")

Clean features   Precision@50 (in-sample): 0.940
+ trend_pct      Precision@50 (in-sample): 1.000   <- confession: leakage, not skill
trend_pct feature-importance share in the leaky model: 1.000

Verdict: trend_pct is excluded from every model below. This block exists only to prove the exclusion is necessary.


**Validation design — client-holdout, not a plain random split.** Pages from the same
client can share a lot of hidden character (CMS quirks, editorial style, niche). A plain random
split lets the model partly memorize the client instead of the pattern, and inflates the score.
We hold out ~20% of *clients* — not rows — so no client's pages appear in both train and test,
and we report the naive split alongside it as an honesty check.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
def metrics_at(y_true, proba, prefix=""):
    pred = (proba >= 0.5).astype(int)
    return {
        f"{prefix}roc_auc": roc_auc_score(y_true, proba) if y_true.nunique() == 2 else float("nan"),
        f"{prefix}avg_precision": average_precision_score(y_true, proba) if y_true.nunique() == 2 else float("nan"),
        f"{prefix}precision_at_50": precision_at_k(y_true, proba, 50),
        f"{prefix}recall": recall_score(y_true, pred, zero_division=0),
        f"{prefix}f1": f1_score(y_true, pred, zero_division=0),
    }

def build_models():
    return {
        "logistic_regression": Pipeline([("scaler", StandardScaler()),
                                          ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
        "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                                 n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    }

X = build_matrix(df)

# --- honest split: client holdout ---
groups = df["client_id"].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])), "client leakage across split!"

# --- naive split: plain stratified row split (the honesty check) ---
naive_train_idx, naive_test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)

results = {}
for split_name, (tr, te) in {"client_holdout": (train_idx, test_idx), "naive_stratified": (naive_train_idx, naive_test_idx)}.items():
    for name, model in build_models().items():
        model.fit(X.iloc[tr], y.iloc[tr])
        proba = model.predict_proba(X.iloc[te])[:, 1]
        results[(split_name, name)] = metrics_at(y.iloc[te], proba)
    baseline_test_scores = df.iloc[te]["baseline_refresh_score"].to_numpy()
    results[(split_name, "baseline_rule")] = metrics_at(y.iloc[te], baseline_test_scores)

rows = []
for (split_name, model_name), m in results.items():
    rows.append({"split": split_name, "model": model_name, **m})
results_table = pd.DataFrame(rows).sort_values(["split", "precision_at_50"], ascending=[True, False])
results_table[["split", "model", "roc_auc", "avg_precision", "precision_at_50", "recall", "f1"]].round(3)


,split,model,roc_auc,avg_precision,precision_at_50,recall,f1
0,client_holdout,logistic_regression,0.616,0.604,0.72,0.627,0.606
2,client_holdout,random_forest,0.610,0.590,0.54,0.601,0.595
1,client_holdout,decision_tree,0.612,0.585,0.50,0.542,0.572
3,client_holdout,baseline_rule,0.498,0.482,0.32,0.248,0.322
5,naive_stratified,decision_tree,0.717,0.701,0.94,0.703,0.697
4,naive_stratified,logistic_regression,0.711,0.727,0.90,0.677,0.677
6,naive_stratified,random_forest,0.758,0.768,0.90,0.732,0.721
7,naive_stratified,baseline_rule,0.579,0.570,0.48,0.456,0.517


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
piv = results_table.pivot(index="model", columns="split", values="precision_at_50")
piv = piv.reindex(["baseline_rule", "logistic_regression", "decision_tree", "random_forest"])
piv[["naive_stratified", "client_holdout"]].plot(kind="bar", ax=ax, color=["#F28E2B", "#4E79A7"])
ax.set_ylabel("Precision@50")
ax.set_title("Precision@50 — honest (client_holdout) vs naive split")
ax.axhline(base_rate, color="gray", linestyle="--", linewidth=1, label=f"base rate ({base_rate:.2f})")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/precision_at_50_split_comparison.png", dpi=150)
plt.close()

best_row = results_table[results_table["split"] == "client_holdout"].sort_values("precision_at_50", ascending=False).iloc[0]
baseline_row = results_table[(results_table["split"] == "client_holdout") & (results_table["model"] == "baseline_rule")].iloc[0]
naive_best = results_table[(results_table["split"] == "naive_stratified") & (results_table["model"] == best_row["model"])].iloc[0]

print(f"Best model (client holdout, honest): {best_row['model']} — Precision@50 = {best_row['precision_at_50']:.3f}")
print(f"Baseline rule (same honest split):                        Precision@50 = {baseline_row['precision_at_50']:.3f}")
lift = best_row['precision_at_50'] / baseline_row['precision_at_50'] if baseline_row['precision_at_50'] > 0 else float('nan')
print(f"Lift over baseline (honest split): {lift:.1f}x")
print(f"\nSame model under a NAIVE stratified split: Precision@50 = {naive_best['precision_at_50']:.3f}")
print(f"Gap between naive and honest split for the best model: {naive_best['precision_at_50'] - best_row['precision_at_50']:+.3f}")
print("That gap is the honesty check — it's the size of the mistake a careless split would have hidden.")

Best model (client holdout, honest): logistic_regression — Precision@50 = 0.720
Baseline rule (same honest split):                        Precision@50 = 0.320
Lift over baseline (honest split): 2.2x

Same model under a NAIVE stratified split: Precision@50 = 0.900
Gap between naive and honest split for the best model: +0.180
That gap is the honesty check — it's the size of the mistake a careless split would have hidden.


**Reading this table:** on this 30k-row starter slice, the client-holdout Precision@50
is the number we stand behind for the paper. Weekly notebook `w08` (`w06_validation_audit.ipynb`)
ran the same audit at warehouse scale and found the same *shape* of result — naive splits
inflate, client-holdout is materially lower and is the trustworthy number — even where the exact
decimals differ from this run. Report the shape, not a borrowed decimal.

In [ ]:
best_model_name = best_row["model"]
full_models = build_models()
full_models[best_model_name].fit(X, y)
best_full_model = full_models[best_model_name]

if hasattr(best_full_model, "feature_importances_"):
    importances = pd.Series(best_full_model.feature_importances_, index=X.columns)
elif isinstance(best_full_model, Pipeline):
    importances = pd.Series(np.abs(best_full_model.named_steps["model"].coef_[0]), index=X.columns)
else:
    importances = pd.Series(np.zeros(X.shape[1]), index=X.columns)

top_features = importances.sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(8, 5))
top_features.sort_values().plot(kind="barh", ax=ax, color="#4E79A7")
ax.set_title(f"Top features — {best_model_name} (fit on all client-holdout-safe data)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance.png", dpi=150)
plt.close()

top_features.round(4)


,0
log_impressions_90d,1.4162
log_clicks_90d,0.6297
word_count,0.5333
avg_position,0.3678
content_age_days,0.3227
char_count,0.3068
impression_tier_low,0.2769
log_sessions_90d,0.2577
position_tier_top_3,0.2049
word_count_tier_1000-2000,0.1827


## 5. Limitations

*What this work cannot claim.*

- **The label is a current-window proxy, not a future outcome.** `is_declining_label` compares
  the last 30 days to the prior 30 days *as of the export date* — it does not predict what a
  page will do next. A stronger capstone-grade label would be "features from days 1–90 predict
  decline in days 91–120," built from the warehouse's daily fact table with a strict leakage
  audit (see `w06_validation_audit.ipynb` and the `hunting-leakage-and-validating` skill).
- **Staleness is a weak-to-reversed signal.** `days_since_last_update` shows the *opposite* of
  the popular belief across every audit run so far (ML-06, ML-07, this notebook's baseline) —
  stale pages decline *less* often, not more. The baseline rule still includes a freshness term
  for transparency and comparability, but this finding means "just refresh old pages" is not,
  by itself, a defensible strategy.
- **This result is one run, on one 30k-row anonymized slice, with fixed seeds.** It is not a
  benchmark on the full ~79M-row warehouse. Directional agreement with the warehouse-scale
  validation audit is noted above; exact decimals are not interchangeable across the two.
- **No causal claim is made or supported.** Nothing here shows that refreshing a page *causes*
  recovery. That would require an experiment or another causal design this data cannot provide.
- **Client-holdout, not time-holdout.** The split generalizes across *clients*, not across
  *time* — it does not test whether the model still works next quarter.
- **Sparse metrics stay sparse.** AI-referral traffic and a few reason codes
  (`stale_visible_page` fires on only a handful of rows in the starter slice) are noted, not
  over-interpreted.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
full_baseline_norm = normalize(df["baseline_refresh_score"])
full_probability = best_full_model.predict_proba(X)[:, 1]
df["model_probability"] = full_probability
df["final_refresh_score"] = (100 * (0.70 * full_probability + 0.30 * full_baseline_norm)).clip(0, 100)

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and ((0 < row["engagement_rate"] < 30) or (0 < row["scroll_rate"] < 30)):
        reasons.append("low_engagement_visible_page")
    if row["model_probability"] >= 0.65:
        reasons.append("model_decline_risk")
    return "|".join(reasons) if reasons else "general_refresh_review"

def suggested_action(reasons):
    r = set(reasons.split("|"))
    if "thin_visible_page" in r:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in r and ("declining_with_demand" in r or "model_decline_risk" in r):
        return "refresh_and_review_ctr"
    if "low_engagement_visible_page" in r and ("declining_with_demand" in r or "model_decline_risk" in r):
        return "refresh_and_review_engagement"
    if r.intersection({"declining_with_demand", "stale_visible_page", "model_decline_risk"}):
        return "refresh"
    return "monitor"

df["final_reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["final_reason_codes"].apply(suggested_action)

queue = df.sort_values(["final_refresh_score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["final_rank"] = queue.index + 1

action_mix = queue["suggested_action"].value_counts()
print("Suggested-action mix across all 30,000 pages:")
print(action_mix.to_string())

queue_cols = ["final_rank", "content_id", "final_refresh_score", "model_probability",
              "suggested_action", "final_reason_codes", "impressions_90d", "sessions_90d",
              "avg_position", "trend_direction"]
top20 = queue[queue_cols].head(20)
top20


Suggested-action mix across all 30,000 pages:
suggested_action
monitor                          14602
refresh_and_review_ctr            7039
refresh                           6336
refresh_and_review_engagement     1941
expand_and_refresh                  82


,final_rank,content_id,final_refresh_score,model_probability,suggested_action,final_reason_codes,impressions_90d,sessions_90d,avg_position,trend_direction
0,1,content_c8e9d6ab9013,94.239596,0.935912,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,208678,6,9.7,down
1,2,content_72fdb385e810,89.724366,0.883624,refresh_and_review_ctr,low_ctr_visible_page|model_decline_risk,19360,5,6.9,stable
2,3,content_825a9788af8d,89.526519,0.881086,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,16786,1,5.6,down
3,4,content_8ba781dafa55,89.042612,0.896011,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,16156,2,9.0,down
4,5,content_5195668f06db,88.842745,0.899469,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,6635,10,5.2,down
5,6,content_e06389cec5c1,88.617587,0.881091,refresh_and_review_ctr,low_ctr_visible_page|model_decline_risk,19572,11,7.0,up
6,7,content_f986bd514b6e,88.473178,0.946904,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,22456,4,6.6,down
7,8,content_fb5ea6f6ea7c,87.094459,0.856834,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,12924,5,7.4,down
8,9,content_2143794a59ca,87.046868,0.836262,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,26413,9,5.5,down
9,10,content_1f080331fa2b,86.784305,0.855616,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,6.8,down


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
action_mix.plot(kind="bar", ax=ax, color="#426B69")
ax.set_title("Suggested action mix — full ranked queue")
ax.set_ylabel("Pages")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/action_mix.png", dpi=150)
plt.close()

queue[queue_cols + ["client_id"]].to_csv(f"{OUT_DIR}/refresh_queue_capstone.csv", index=False)
print(f"Wrote {OUT_DIR}/refresh_queue_capstone.csv ({len(queue):,} rows)")


Wrote work/outputs/refresh_queue_capstone.csv (30,000 rows)


**Human review and the no-go list.** This queue is a reviewer aid, not an auto-publish
trigger. A reviewer should still open each high-priority page before acting. Two hard no-gos:
never auto-apply `expand_and_refresh` to a page with `thin_visible_page` alone if the low word
count reflects the content *type* (e.g. a short FAQ) rather than genuine thinness; never treat
`model_decline_risk` on a page with `impressions_90d` near the reporting floor as equivalent to
a high-traffic page's decline — check volume before acting on any single-digit-impression row.

**Monitoring / staleness triggers.** Re-run this notebook's Section 4 whenever the model's
feature distributions drift meaningfully from this run (e.g. quarterly), and treat a
client-holdout Precision@50 drop of more than ~0.10 from this baseline as a signal to
re-audit for leakage before trusting the queue again.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
summary = {
    "rows_scored": int(len(df)),
    "clients": int(df["client_id"].nunique()),
    "declining_label_rate": float(base_rate),
    "best_model": best_model_name,
    "client_holdout_precision_at_50": float(best_row["precision_at_50"]),
    "baseline_precision_at_50_client_holdout": float(baseline_row["precision_at_50"]),
    "lift_over_baseline_honest_split": float(lift),
    "naive_split_precision_at_50_best_model": float(naive_best["precision_at_50"]),
    "naive_vs_honest_gap": float(naive_best["precision_at_50"] - best_row["precision_at_50"]),
    "leaky_trend_pct_precision_at_50_in_sample": float(leaky_p50),
    "clean_precision_at_50_in_sample": float(clean_p50),
    "top_features": top_features.round(4).to_dict(),
    "action_mix": action_mix.to_dict(),
    "top20_preview": top20.to_dict(orient="records"),
}
with open(f"{OUT_DIR}/capstone_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("Wrote", f"{OUT_DIR}/capstone_summary.json")
print("Figures written to", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))


Wrote work/outputs/capstone_summary.json
Figures written to work/figures : ['action_mix.png', 'precision_at_50_split_comparison.png', 'top_feature_importance.png']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 8. Demo outline (5 minutes) + shareable cuts

### 5-minute demo outline

**Question (30s):** FlyRank's content already decays after publishing — rankings slip,
clicks drop, and teams notice too late. With capacity to review only ~50 of ~30,000 pages a
week, which ones should a reviewer look at first?

**Method (90s):** Built a transparent baseline rule first (visibility + staleness + position +
depth, no fitted weights), then a small model comparison (logistic regression, decision tree,
random forest) on 18 numeric + 8 categorical observable features. No product decision flags,
no label-derived columns — checked directly with a leakage test. Evaluated with a
client-holdout split (GroupShuffleSplit on `client_id`), reported next to a naive split as an
honesty check.

**One chart (60s):** [Precision@50 by model, honest vs naive split] — show the bar chart from
Section 4. The naive split says 0.90; the honest client-holdout split says 0.72. That 0.18-point
gap is the demo's punchline: a careless validation choice would have quietly overstated the
result by 25%.

**One honest result (60s):** On the honest split, the best model reaches Precision@50 of 0.72
versus 0.32 for the hand-written rule — a 2.2x lift. Also: staleness turned out to be a weak-to-
reversed signal across every audit this track ran, which overturned my own starting intuition
("stale pages decline less often, not more").

**One recommendation (60s):** Ship the ranked queue as a reviewer aid with reason codes
(`declining_with_demand`, `low_ctr_visible_page`, `model_decline_risk`), not an auto-publish
trigger. Top of the queue skews toward `refresh_and_review_ctr` — pages with real demand and
weak click-through at their position, which is a concrete, explainable place to start.

---

### Shareable cut 1 — social post (methodology)

Spent 8 weeks building a content-refresh priority model for FlyRank's ML internship —
here's the one methodology lesson that mattered most.

I trained a model to rank ~30,000 pages by "should a reviewer look at this first?" A naive
train/test split said Precision@50 = 0.90. Nice number. Wrong number.

The dataset has 32 clients, and pages from the same client share hidden structure — writing
style, CMS quirks, niche. A naive random split lets the model partly memorize the client
instead of learning the pattern. Once I held out entire clients instead of rows
(client-holdout, not random), the honest number was 0.72 — still a real 2.2x lift over a
transparent hand-written baseline (0.32), but 0.18 points lower than the flattering number.

The gap between a naive split and a client-holdout split is itself a finding, and reporting
both, side by side, is what separates a real result from a lucky one.

Repo: github.com/LaibaSabir1/flyrank-ml-internship-laiba_sabir

---

### Shareable cut 2 — employer-facing summary (3 sentences)

I built and validated a content-refresh priority model on FlyRank's anonymized search
performance dataset (30,000 pages, 32 clients), replacing an ad-hoc review order with a
ranked, reason-coded queue. Using a client-holdout validation split — the honest choice given
that pages from the same client share hidden structure — the model beat a transparent
hand-written baseline by 2.2x on Precision@50 (0.72 vs 0.32), while I separately confirmed
via a leakage audit and a naive-vs-honest split comparison that the result wasn't an artifact
of validation choices. The deliverable is a public research paper with full methodology,
limitations, and a ranked action playbook, plus a reproducible notebook and repo.